# ARVO Router 학습과 OpenRouter 평가

합성 데이터는 사용하지 않습니다. ARVO의 fix patch와 crash stack으로 만든 독립 ground truth를 사용하고, 프로젝트 단위로 분리된 train/dev/test에서 Router를 학습·평가합니다. OpenRouter LLM은 fine-tuning하지 않고 취약점 분석 추론에만 사용합니다.

In [ ]:
import json
from pathlib import Path
from pprint import pprint
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
DATA_DIR = ROOT / 'data' / 'arvo'

from llm_security.config import AppConfig
from llm_security.datasets import load_cases_jsonl, load_router_samples_jsonl
from llm_security.experiment import ExperimentRunner
from llm_security.factory import build_pipeline
from llm_security.models import to_dict
from llm_security.router import LearnedRouter, train_and_evaluate_router

## 1. ARVO 데이터와 project-disjoint split 확인

`prepare-arvo`가 생성한 manifest에서 프로젝트 중복 여부와 Expert family 분포를 확인합니다.

In [ ]:
manifest_path = DATA_DIR / 'manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(
        'ARVO processed data is missing. Run: python -m llm_security.cli prepare-arvo'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
pprint(manifest)
project_sets = {
    split: set(details['projects'])
    for split, details in manifest['splits'].items()
}
assert project_sets['train'].isdisjoint(project_sets['dev'])
assert project_sets['train'].isdisjoint(project_sets['test'])
assert project_sets['dev'].isdisjoint(project_sets['test'])
print('Project leakage: none')

## 2. 실제 ARVO Router 학습

학습은 ARVO train split만 사용합니다. Dev는 모델 선택 지표, test는 최종 일반화 지표로 한 번만 확인합니다.

In [ ]:
config = AppConfig.from_env(ROOT / '.env')
train_samples = load_router_samples_jsonl(DATA_DIR / 'router_train.jsonl')
dev_samples = load_router_samples_jsonl(DATA_DIR / 'router_dev.jsonl')
test_samples = load_router_samples_jsonl(DATA_DIR / 'router_test.jsonl')
router, dev_metrics = train_and_evaluate_router(
    train_samples,
    dev_samples,
    threshold=config.router.threshold,
    max_experts=config.router.max_experts,
    seed=config.runtime.seed,
)
test_metrics = router.evaluate(test_samples)
artifact = ROOT / 'models' / 'router-arvo.pkl'
router.save(artifact)
print('train/dev/test samples:', len(train_samples), len(dev_samples), len(test_samples))
print('router artifact:', artifact)
print('dev metrics:')
pprint(to_dict(dev_metrics))
print('test metrics:')
pprint(to_dict(test_metrics))

## 3. Test split 예측 확인

보지 못한 프로젝트의 취약 후보에 대해 정답 family와 선택 Expert를 비교합니다.

In [ ]:
for sample in test_samples:
    decision = router.route(sample.candidate)
    expected = [family.value for family in sample.labels]
    selected = [family.value for family in decision.selected]
    top_scores = sorted(
        ((family.value, round(score, 4)) for family, score in decision.scores.items()),
        key=lambda item: item[1],
        reverse=True,
    )[:3]
    print(sample.candidate.project_id, sample.candidate.file, sample.candidate.function)
    print('  expected:', expected)
    print('  selected:', selected)
    print('  top scores:', top_scores)

## 4. 실제 ARVO test benchmark

`.env`의 `RUN_PAID_EXPERIMENTS=1`일 때만 test 프로젝트를 OpenRouter로 분석합니다. 모델 sweep은 같은 모델을 중복 호출하므로 이 노트북에서 제거했습니다.

In [ ]:
print('API key configured:', bool(config.model.api_key))
print('Expert model:', config.model.expert_model)
print('Validator model:', config.model.validator_model)

if config.runtime.allow_paid_experiments:
    if not config.model.api_key:
        raise RuntimeError('Set OPENROUTER_API_KEY in .env first.')
    test_cases = load_cases_jsonl(DATA_DIR / 'cases_test.jsonl')
    runner = ExperimentRunner(build_pipeline(config, router))
    experiment = runner.run(test_cases)
    runner.save(experiment, ROOT / 'experiment-arvo.json')
    pprint(to_dict(experiment.aggregate))
else:
    print('OpenRouter run skipped: RUN_PAID_EXPERIMENTS=0')